# IDR public dataset demo

The Image Data Resource (IDR — https://idr.openmicroscopy.org) hosts public image datasets including cleared-tissue light-sheet acquisitions, served as OME-Zarr on S3.

This notebook runs `volumetric-qc` against one such dataset by streaming directly from the public S3 bucket — no full download required.

**Note.** Network access required. The exact IDR study ID below may rotate; if it fails, browse https://idr.openmicroscopy.org/about/img_links.html for a current cleared-tissue dataset and update the URL.

In [ ]:
# Example IDR cleared-tissue light-sheet OME-Zarr URL. Update if rotated.
IDR_ZARR = 's3://idr/zarr/v0.4/idr0101A/13457537.zarr'  # ENDOC / iDISCO-cleared whole-mount sample

In [ ]:
import zarr
import fsspec

# Anonymous public-bucket access.
store = zarr.storage.FSStore(IDR_ZARR, anon=True)
root = zarr.open(store, mode='r')
print(list(root.array_keys())[:5], list(root.group_keys())[:5])

## Run QC against the highest-resolution multiscale level

OME-Zarr stores typically have a multiscale pyramid (0 = highest resolution, N-1 = lowest). For QC we usually want the highest resolution; for a fast smoke test pick a lower level.

In [ ]:
from volumetric_qc import open_volume, run_qc, load_preset
from volumetric_qc.reports import write_html_report, write_json_summary

# open_volume auto-picks the highest-res level. For a smoke test you can pass
# the pre-downsampled level directly: zarr.open(store, mode='r')['3']
lv = open_volume(IDR_ZARR)
print('shape:', lv.shape, 'voxel_size_um:', lv.voxel_size_um)
print('channels:', lv.channel_names)

In [ ]:
cfg = load_preset('idisco')
cfg.sampling.z_stride = 4         # IDR volumes are deep — subsample for a fast smoke test
cfg.sampling.xy_downsample = 2
cfg.sampling.blob_z_sample = 8
result = run_qc(lv, cfg, progress=True)
print(f'\noverall_pass: {result.overall_pass}  n_fail: {result.n_fail}  n_warn: {result.n_warn}')

In [ ]:
write_html_report(result, 'idr_demo.html', title=f'IDR cleared-tissue QC — {IDR_ZARR.split("/")[-1]}')
write_json_summary(result, 'idr_demo.json')

## Inspecting flags

The first IDR cleared-tissue sample you point this at will probably surface real, interpretable QC findings — the dataset is not synthetic, the artifacts are not synthetic, the thresholds were set against published-grade data.

In [ ]:
for f in result.flags:
    if f.severity != 'pass':
        print(f'{f.severity:5s}  {f.name:40s}  value={f.value:.4f}  thr={f.threshold:.2f}')
        print(f'         {f.message}')